# CPSC 452 Final Project

## Four-model comparison (linear, symbolic, diffusion, structured energy)

This notebook runs an end-to-end comparison on the **same trajectory data** for each environment:

- **Model 1 — Linear model (baseline)**: `PolynomialConservation` (learns a linear combination of fixed features; yields an explicit equation)
- **Model 2 — Symbolic learning (no hand-fed library)**: `PySR` symbolic regression fit to a *learned invariant signal* (from Model 4), so it must infer the functional form from data
- **Model 3 — Diffusion model**: conditional diffusion over one-step transitions, rolled out to full trajectories
- **Model 4 — Energy-guided model (K + V prior)**: `StructuredEnergyNetwork` with inductive bias \(H(q,v)=T(v)+V(q)\)

Outputs are saved to `models/`, `figures/`, and `results/`.


## Setup

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")

# Robust project root detection:
# - If notebook is launched from repo root, use cwd
# - If launched from notebooks/, use parent
cwd = os.path.abspath(os.getcwd())
PROJECT_ROOT = os.path.abspath(os.path.join(cwd, "..")) if os.path.basename(cwd) == "notebooks" else cwd

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
FIGURES_DIR = os.path.join(PROJECT_ROOT, "figures")

# Ensure imports work when running from notebooks/
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Models dir:", MODELS_DIR)
print("Figures dir:", FIGURES_DIR)
print("Torch:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

## Day 1 — Data generation (optional) + load

In [ ]:
from src.data_generation.projectile import generate_projectile_data
from src.data_generation.pendulum import generate_pendulum_data
from src.data_generation.spring_mass import generate_spring_mass_data

# Toggle to regenerate .npy files (otherwise we just load existing)
REGENERATE_DATA = False

proj_path = os.path.join(DATA_DIR, "projectile", "trajectories.npy")
pend_path = os.path.join(DATA_DIR, "pendulum", "trajectories.npy")
spring_path = os.path.join(DATA_DIR, "spring_mass", "trajectories.npy")

if REGENERATE_DATA:
    os.makedirs(os.path.join(DATA_DIR, "projectile"), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "pendulum"), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "spring_mass"), exist_ok=True)

    proj = generate_projectile_data()
    np.save(proj_path, proj)

    pend = generate_pendulum_data()
    np.save(pend_path, pend)

    spring = generate_spring_mass_data()
    np.save(spring_path, spring)

proj_raw = np.load(proj_path)
pend_raw = np.load(pend_path)
spring_raw = np.load(spring_path)

print("Projectile:", proj_raw.shape, proj_raw.dtype)
print("Pendulum:", pend_raw.shape, pend_raw.dtype)
print("Spring-mass:", spring_raw.shape, spring_raw.dtype)

## Sanity checks (raw space)

We run everything on **raw trajectories** for fairness (all models see the same data). Here we just verify the synthetic trajectories are energy-conserving in raw space.

In [ ]:
from src.data_generation.projectile import compute_energy_projectile
from src.data_generation.pendulum import compute_energy_pendulum
from src.data_generation.spring_mass import compute_energy_spring

E_proj = compute_energy_projectile(proj_raw)
E_pend = compute_energy_pendulum(pend_raw)
E_spring = compute_energy_spring(spring_raw)

print("Projectile mean within-trajectory energy std:", float(E_proj.std(axis=1).mean()))
print("Pendulum mean within-trajectory energy std:", float(E_pend.std(axis=1).mean()))
print("Spring-mass mean within-trajectory energy std:", float(E_spring.std(axis=1).mean()))

## Train + evaluate the four models

In [ ]:
from dataclasses import asdict

from src.data_generation.utils import train_val_split
from src.evaluation.symbolic_regression import pysr_available, discover_equation_pysr
from src.evaluation.validate_cdn import validate_conservation_model
from src.evaluation.validate_diffusion import evaluate_diffusion_rollout
from src.models.polynomial_cdn import PolynomialConservation
from src.training.train_diffusion import DiffusionTrainConfig, train_diffusion_transition
from src.training.train_polynomial import train_polynomial_model
from src.training.train_structured_energy import StructuredEnergyConfig, train_structured_energy

# Ensure outputs go to the project folders even when executed from notebooks/
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_ROOT, "results"), exist_ok=True)
os.chdir(PROJECT_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def fit_env_four_models(
    env_name: str,
    trajs_raw: np.ndarray,
    energy_fn,
    state_dim: int,
    var_names: list[str],
    pos_dims: list[int],
    vel_dims: list[int],
):
    print("\n" + "=" * 80)
    print("ENV:", env_name)
    print("trajs:", trajs_raw.shape)

    trajs_train, trajs_val = train_val_split(trajs_raw, val_fraction=0.1, seed=42)

    # ------------------------------
    # Model 4: Structured energy (K+V)
    # ------------------------------
    E0_train = energy_fn(trajs_train)[:, 0].astype(np.float32)
    structured_cfg = StructuredEnergyConfig(
        env_name=env_name,
        state_dim=state_dim,
        pos_dims=pos_dims,
        vel_dims=vel_dims,
        device=str(device),
        save_dir=MODELS_DIR,
        epochs=300,
        batch_size=1024,
        lr=1e-3,
        hidden_dim=128,
        n_layers=2,
        lambda_var=1.0,
        lambda_energy=0.1,
    )
    structured_model, _ = train_structured_energy(trajs_train, energy0_np=E0_train, cfg=structured_cfg)
    structured_metrics = validate_conservation_model(
        structured_model,
        trajs_val,
        energy_fn,
        env_name,
        model_name="StructuredEnergy",
        save_dir=FIGURES_DIR,
        device=device,
    )

    # ------------------------------
    # Model 1: Linear (polynomial) baseline
    # ------------------------------
    E0_all = energy_fn(trajs_raw)[:, 0].astype(np.float32)
    poly_model, poly_eq = train_polynomial_model(
        trajs_train,
        state_dim=state_dim,
        env_name=env_name,
        var_names=var_names,
        energy0_np=E0_all,
        save_dir=MODELS_DIR,
        degree=2,
        include_trig_dims=([0] if env_name == "pendulum" else None),
        device=device,
        epochs=1200,
        warmup_epochs=150,
    )
    poly_metrics = validate_conservation_model(
        poly_model,
        trajs_val,
        energy_fn,
        env_name,
        model_name="LinearPolynomial",
        save_dir=FIGURES_DIR,
        device=device,
    )

    # ------------------------------
    # Model 3: Diffusion rollout model
    # ------------------------------
    diff_cfg = DiffusionTrainConfig(
        env_name=env_name,
        state_dim=state_dim,
        device=str(device),
        save_dir=MODELS_DIR,
        epochs=25,
        lr=2e-4,
        batch_size=4096,
        K=50,
        hidden_dim=256,
        time_emb_dim=64,
    )
    diff_model, diff_stats, _ = train_diffusion_transition(trajs_train, cfg=diff_cfg)
    diff_metrics = evaluate_diffusion_rollout(
        model=diff_model,
        trajs_raw=trajs_val,
        energy_fn_np=energy_fn,
        env_name=env_name,
        stats=diff_stats,
        n_rollouts=256,
        device=str(device),
        save_dir=FIGURES_DIR,
    )

    # ------------------------------
    # Model 2: Symbolic learning (PySR) from learned invariant signal
    #   We fit PySR to (state -> structured_model(state)), not to analytical energy.
    # ------------------------------
    symbolic = {"available": bool(pysr_available()), "equation": None}
    if pysr_available():
        flat = trajs_train.reshape(-1, state_dim).astype(np.float32)
        rng = np.random.RandomState(0)
        n = min(50000, flat.shape[0])
        X = flat[rng.choice(flat.shape[0], size=n, replace=False)].astype(np.float64)

        with torch.no_grad():
            y = structured_model(torch.tensor(X, dtype=torch.float32, device=device)).detach().cpu().numpy().astype(np.float64)

        reg = discover_equation_pysr(
            X=X,
            y=y,
            variable_names=var_names,
            binary_operators=["+", "-", "*", "/"],
            unary_operators=(["square", "sin", "cos"] if env_name == "pendulum" else ["square"]),
            niterations=200,
            model_selection="best",
        )
        try:
            symbolic["equation"] = str(reg.sympy())
        except Exception:
            symbolic["equation"] = str(reg)

    return {
        "env": env_name,
        "linear_equation": poly_eq,
        "linear_metrics": poly_metrics,
        "structured_cfg": asdict(structured_cfg),
        "structured_metrics": structured_metrics,
        "diffusion_cfg": asdict(diff_cfg),
        "diffusion_metrics": diff_metrics,
        "symbolic": symbolic,
    }


results = {}

results["projectile"] = fit_env_four_models(
    env_name="projectile",
    trajs_raw=proj_raw,
    energy_fn=compute_energy_projectile,
    state_dim=4,
    var_names=["x", "y", "vx", "vy"],
    pos_dims=[0, 1],
    vel_dims=[2, 3],
)

results["pendulum"] = fit_env_four_models(
    env_name="pendulum",
    trajs_raw=pend_raw,
    energy_fn=compute_energy_pendulum,
    state_dim=2,
    var_names=["theta", "omega"],
    pos_dims=[0],
    vel_dims=[1],
)

results["spring_mass"] = fit_env_four_models(
    env_name="spring_mass",
    trajs_raw=spring_raw,
    energy_fn=compute_energy_spring,
    state_dim=2,
    var_names=["x", "v"],
    pos_dims=[0],
    vel_dims=[1],
)

# save a lightweight JSON summary
import json

out_path = os.path.join(PROJECT_ROOT, "results", "four_model_results.json")
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

print("\nSaved:", out_path)
print("\nSymbolic equations (if available):")
for k, v in results.items():
    print("-", k, ":", v["symbolic"]["equation"])

## Notes on the “symbolic learning” requirement

- The **linear model** (`PolynomialConservation`) is *not* symbolic learning—it’s a fixed hand-designed feature library.
- The **symbolic model** here uses `PySR` to infer a closed-form expression from data.
- To avoid “hand-feeding the true energy equation”, we fit PySR to the **learned invariant output** of the structured \(T+V\) model: \(s \mapsto \hat H(s)\).

If you *do* want symbolic regression directly on \(E(s)\) (the analytical function), that is easy too—but it is inherently using hand-provided physics in the label.


In [ ]:
# Optional: if PySR isn't available, you can still run the other 3 models.
# PySR requires Julia installed and the `pysr` Python package.
#
# In this repo, `requirements.txt` already includes `pysr>=0.16.0`, but Julia setup is external.

## Outputs

After running this notebook you should have:

- **Models** (saved in `models/`)
  - `polynomial_<env>_best.pt`
  - `structured_energy_<env>_best.pt`
  - `diffusion_<env>_best.pt`

- **Figures** (saved in `figures/`)
  - Conservation scatter plots for the linear + structured models (via `validate_conservation_model`)
  - Diffusion rollout metrics JSONs (plus console summary)

- **Summary JSON**
  - `results/four_model_results.json`
